# Register the v1 NRDS forecast datasets and copy their legacy history

Sets `public` up for the `v1-ingest-nrds-*` flows (`prefect-workflows/v1/flows/ingest_nrds.py`):

| v1 (`public`) | what |
| --- | --- |
| `nrds_v22_cfenom_short_range`, `nrds_v22_cfenom_medium_range`, `nrds_v22_lstm0_short_range`, `nrds_v22_lstm0_medium_range` | the NextGen Research DataStream's CFE-NOM and LSTM_0 forecasts (hydrofabric v2.2), legacy's configurations of those names |
| `nrds_v22_cfenom_short_range_series` | the series table they share (`nrds22-<feature_id>`, legacy's ids) |
| `usgs_nrds22` (crosswalk) | the USGS gages to their nrds22 reaches (legacy's crosswalk): pairs `usgs_inst_streamflow` with every dataset above, and is where the ingests find their reaches |

Values keep t-route's times, as legacy and the ingests do (a cycle's first value is two hours
after it).

```yaml
inputs:
  primary: {dataset: public.usgs_inst_streamflow}
  secondary: [{dataset: public.nrds_v22_cfenom_short_range}]
pair: {relation: public.usgs_nrds22}
```

**Run it as a platform admin, on use1, before the ingests' deployments** (their schedules need
the datasets and the crosswalk, and a month copied replaces what an ingest wrote in it).
Spark writes the rows straight into the `public` Iceberg tables (as `06_nwm_points`).
**Re-runnable:** registering skips what exists; the crosswalk's links are written only while
it's empty; series are added, never changed; each month copied replaces that month. Start
with **2. Count**: it writes nothing.

In [ ]:
import os

import pandas as pd
import requests
import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

teehr.__version__

## 1. Settings

In [ ]:
#: v1 dataset (legacy's configuration of the same name) -> its model and configuration.
DATASETS = {
    "nrds_v22_cfenom_short_range": ("CFE-NOM", "short range"),
    "nrds_v22_cfenom_medium_range": ("CFE-NOM", "medium range (member 1)"),
    "nrds_v22_lstm0_short_range": ("LSTM_0", "short range"),
    "nrds_v22_lstm0_medium_range": ("LSTM_0", "medium range (member 1)"),
}
# The first registered makes the series table (`<name>_series`); the rest share it.
ANCHOR = "nrds_v22_cfenom_short_range"
SERIES = f"{ANCHOR}_series"
VERSION = "nrds22"  # the reaches' key prefix
CROSSWALK, OBSERVED = f"usgs_{VERSION}", "usgs_inst_streamflow"
# Datasets whose legacy history to copy: all of them, by default. Narrow it to re-copy some.
COPY = list(DATASETS)
MAX_LEAD = "12D"  # longer than any forecast's last value (medium range: 10 days and 2 hours)
SOURCE = ("NextGen Research DataStream (s3://ciroh-community-ngen-datastream, t-route output, "
          "hydrofabric v2.2); history before the v1 ingests copied from the legacy TEEHR warehouse")

CONTROL_PLANE = os.environ.get(
    "CONTROL_PLANE_URL", "http://teehr-control-plane.teehr-hub.svc.cluster.local:8000"
)
VERIFY_TLS = "local.app.garden" not in CONTROL_PLANE  # KinD's certificate is self-signed
LEGACY, PUBLIC = "iceberg.teehr", "iceberg.public"

In [ ]:
# A cluster for the copy (5.2 B rows since 2025-11); local mode is enough to count and register
# (comment the arguments out).
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=16,
    executor_memory="16g",
    executor_cores=4,
)
spark.conf.set("spark.sql.iceberg.check-nullability", "false")
spark.conf.set("spark.sql.session.timeZone", "UTC")  # months are UTC, as Iceberg's partitions


def access_token():
    """Your access token, refreshed when it's about to expire (JupyterHub sets it)."""
    try:
        from teehr.utils.keycloak_auth import refresh_user_token_from_env
    except ImportError:  # a kernel without the helper: the token as JupyterHub set it
        return os.environ["POLARIS_USER_TOKEN"]
    access, _refresh = refresh_user_token_from_env()
    return access


api = requests.Session()
api.verify = VERIFY_TLS


def call(method, path, **kwargs):
    resp = api.request(method, f"{CONTROL_PLANE}/v1{path}",
                       headers={"Authorization": f"Bearer {access_token()}"}, **kwargs)
    if resp.status_code >= 400 and resp.status_code != 404:
        raise RuntimeError(f"{method} {path}: HTTP {resp.status_code}: {resp.text[:1000]}")
    return resp


def exists(name):
    return call("GET", f"/projects/public/datasets/{name}").status_code == 200


def register(body):
    if exists(body["name"]):
        print(f"public.{body['name']}: exists")
        return
    call("POST", "/projects/public/datasets", json=body)
    print(f"public.{body['name']}: registered")


def month_range(column, month, lead="0D"):
    """A time range filter for one UTC month (prunes the scan), widened by ``lead``."""
    start = pd.Period(month, freq="M")
    end = (start + 1).start_time + pd.Timedelta(lead)
    return (f"{column} >= TIMESTAMP '{start.start_time:%Y-%m-%d} 00:00:00' "
            f"AND {column} < TIMESTAMP '{end:%Y-%m-%d %H:%M:%S}'")


def legacy_rows(dataset):
    return (f"{LEGACY}.secondary_timeseries WHERE configuration_name = '{dataset}' "
            "AND value IS NOT NULL")


me = call("GET", "/projects/public").json()
print(f"public: {me['status']}; your role there: {me.get('my_role')}")

## 2. Count (writes nothing)

Legacy's rows per dataset (one variable and unit each, no members), and per month of
`reference_time`: what the copy writes, and the check compares.

In [ ]:
%%time
names = ", ".join(f"'{d}'" for d in DATASETS)
spark.sql(f"""
    SELECT configuration_name, variable_name, unit_name, count(*) AS rows,
           count(DISTINCT location_id) AS reaches, count(member) AS with_member,
           min(reference_time) AS first, max(reference_time) AS last
    FROM {LEGACY}.secondary_timeseries
    WHERE configuration_name IN ({names}) AND value IS NOT NULL GROUP BY 1, 2, 3 ORDER BY 1
""").show(truncate=False)
legacy = spark.sql(f"""
    SELECT configuration_name AS dataset, date_format(reference_time, 'yyyy-MM') AS month,
           count(*) AS legacy
    FROM {LEGACY}.secondary_timeseries
    WHERE configuration_name IN ({names}) AND value IS NOT NULL GROUP BY 1, 2 ORDER BY 1, 2
""").toPandas()
legacy.pivot(index="month", columns="dataset", values="legacy")

## 3. Register the datasets and the crosswalk (as you)

Forecasts of instantaneous streamflow in m3/s (legacy's `m^3/s`: the counts above must show
only that unit), no `member`. The anchor first: it makes the series table the others share.

In [ ]:
def body(dataset):
    model, configuration = DATASETS[dataset]
    return {
        "type": "timeseries", "name": dataset, "role": "forecast",
        "variable": "streamflow", "statistic": "inst",
        **({} if dataset == ANCHOR else {"series_table": SERIES}),
        "source": SOURCE,
        "description": (f"NRDS {model} {configuration} streamflow forecasts at the gages' reaches "
                        "(NextGen, hydrofabric v2.2): t-route's times, a cycle's first two hours "
                        "after it"),
    }


register(body(ANCHOR))
for dataset in DATASETS:
    if dataset != ANCHOR:
        register(body(dataset))
register({"type": "crosswalk", "name": CROSSWALK, "left": OBSERVED, "right": ANCHOR,
          "source": "Legacy TEEHR warehouse (location crosswalks)",
          "description": f"USGS gages to their {VERSION} (NRDS, hydrofabric v2.2) reaches"})

## 4. The crosswalk's links and the series

The links from legacy's crosswalk, only while the crosswalk is empty. Every linked reach is
then added to the series table (missing ones only), so the series are there before any rows.

In [ ]:
%%time
if spark.table(f"{PUBLIC}.{CROSSWALK}").limit(1).count() == 0:
    spark.sql(f"""
        SELECT DISTINCT primary_location_id AS left_key, secondary_location_id AS right_key
        FROM {LEGACY}.location_crosswalks
        WHERE primary_location_id LIKE 'usgs-%' AND secondary_location_id LIKE '{VERSION}-%'
    """).writeTo(f"{PUBLIC}.{CROSSWALK}").append()
print(f"{CROSSWALK}: {spark.table(f'{PUBLIC}.{CROSSWALK}').count():,} links")

spark.sql(f"""
    MERGE INTO {PUBLIC}.{SERIES} t
    USING (SELECT DISTINCT right_key AS series_key FROM {PUBLIC}.{CROSSWALK}) s
    ON t.series_key = s.series_key
    WHEN NOT MATCHED THEN INSERT (series_key, location_key, location_layer, period, resolution)
        VALUES (s.series_key, NULL, NULL, NULL, NULL)
""")
print(f"{SERIES}: {spark.table(f'{PUBLIC}.{SERIES}').count():,} series")

## 5. Copy the history

A month of `reference_time` at a time, each replacing that month of the v1 table. Values are
legacy's float32 (the files' own), as doubles: what the ingests write.

In [ ]:
START = None  # e.g. ("nrds_v22_lstm0_short_range", "2026-10") to resume from a dataset's month


def copy_month(dataset, month):
    """Replace one UTC month (of reference_time) of the v1 dataset with the legacy rows."""
    # Legacy is partitioned by month of value_time: bound it too, so the scan reads two months
    # of partitions, not all.
    spark.sql(f"""
        SELECT location_id AS series_key, reference_time, value_time,
               CAST(value AS DOUBLE) AS value
        FROM {legacy_rows(dataset)}
          AND {month_range("reference_time", month)}
          AND {month_range("value_time", month, MAX_LEAD)}
    """).writeTo(f"{PUBLIC}.{dataset}").overwritePartitions()

In [ ]:
%%time
for dataset, month in legacy[["dataset", "month"]].itertuples(index=False):
    if dataset not in COPY or (START and (dataset, month) < START):
        continue
    copy_month(dataset, month)
    print(f"{dataset} {month}: copied")

## 6. Check: v1 against legacy, month by month

In [ ]:
%%time
v1 = pd.concat([spark.sql(f"""
    SELECT '{d}' AS dataset, date_format(reference_time, 'yyyy-MM') AS month, count(*) AS v1
    FROM {PUBLIC}.{d} GROUP BY 2
""").toPandas() for d in COPY])
both = legacy[legacy.dataset.isin(COPY)].merge(v1, on=["dataset", "month"], how="outer").fillna(0)
both["matches"] = both.legacy == both.v1
for dataset, rows in both.groupby("dataset"):
    last = spark.sql(f"SELECT max(reference_time) AS t FROM {PUBLIC}.{dataset}").first().t
    print(f"{dataset}: {int(rows.v1.sum()):,} rows; every month matches: {rows.matches.all()}; "
          f"last cycle: {last}")
both[~both.matches]

**After it:** merge the ingests' PR (their schedules are on). Each first run reads from its
dataset's last copied cycle on; legacy keeps ingesting until the two match.

In [ ]:
spark.stop()